# Stratégie de mean reversion entre 2 Actifs Corrélés mais non stationnaires

https://blog.quantinsti.com/statistical-arbitrage/

Nous allons mettre en place une stratégie d'arbitrage statistique entre 2 actifs très corrélés mais non stationnaires comme l'or ou l'argent et comparé le resutlats avec des actifs stationnaires comme .... . L'hypothèse c'est que des anomalies de marché peuvent se produire et décoreller temporairement les 2 actifs mais que ça va finir par revenir à la normale. Miser sur le retour à la normale en achetant un actif et en shortant le 2ème peut être rentable.


In [1]:
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.stattools import adfuller


In [2]:

# Conf download 
INTERVAL = '1d'
PERIOD = '5y'  

print(f"Téléchargement des données {INTERVAL} (période: {PERIOD})...")

or_data = yf.download('GLD', period=PERIOD, interval=INTERVAL, progress=False)
argent_data = yf.download('SLV', period=PERIOD, interval=INTERVAL, progress=False)


df = pd.DataFrame({
    'Or': or_data['Close'].squeeze(),
    'Argent': argent_data['Close'].squeeze()
}).dropna()

print(f"\nDonnées chargées:")
print(f"{len(df)} points")
print(df.tail(1))

# Plotly du dataframe
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Or'],
    mode='lines',
    name='Or (GLD)',
    line=dict(color='gold', width=2)
))

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Argent'],
    mode='lines',
    name='Argent (SLV)',
    line=dict(color='silver', width=2)
))

fig.update_layout(
    title='Évolution des cours de l\'Or et de l\'Argent (close daily)',
    xaxis_title='Date',
    yaxis_title='Prix ($)',
    hovermode='x unified',
    template='plotly_white',
    height=600
)

fig.show()


Téléchargement des données 1d (période: 5y)...

Données chargées:
1256 points
                    Or     Argent
Date                             
2026-03-23  404.350006  61.955002


Pour notre stratégie nous avons besoin de 2 actifs corrélés. On va tester si l'or et l'argent le sont.


In [4]:
# Vérification de la corrélation avec la matrice de Pearson
correlation_matrix = df.corr(method='pearson')
correlation_value = correlation_matrix.loc['Or', 'Argent']

# Visualisation de la matrice de corrélation
fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.index,
    colorscale='RdBu',
    zmid=0,
    text=correlation_matrix.values.round(4),
    texttemplate='%{text}',
    textfont={"size": 14},
    colorbar=dict(title="Corrélation")
))

fig.update_layout(
    title='Matrice de Corrélation de Pearson',
    height=400,
    template='plotly_white'
)

fig.show()


Les 2 actifs sont bien corrélés.

Cependant pour notre stratégie d'arbitrage nous devons avoir 2 actifs comparables. Ici le prix de l'or et de l'argent est trop différent, nous devons donc calculer un ratio pour que 1x Or = ratio * Argent.


In [3]:
# Régression linéaire pour calculer le ratio entre l'or et l'argent
# On modélise: Or = ratio * Argent + intercept

X = df[['Argent']].values
y = df['Or'].values

# Régression linéaire
reg = LinearRegression()
reg.fit(X, y)

ratio = reg.coef_[0]
intercept = reg.intercept_

print(f"Résultats de la régression linéaire:")
print(f"Ratio (coefficient): {ratio:.4f}")
print(f"R²: {reg.score(X, y):.4f}")

# Calcul de l'argent ajusté avec le ratio
df['Argent_ajuste'] = df['Argent'] * ratio + intercept


Résultats de la régression linéaire:
Ratio (coefficient): 5.4821
R²: 0.8511


In [4]:
#tracée des cours avec l'argent ajusté du ratio
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Or'],
    mode='lines',
    name='Or (GLD)',
    line=dict(color='gold', width=2)
))

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Argent_ajuste'],
    mode='lines',
    name=f'Argent ajusté (ratio={ratio:.4f})',
    line=dict(color='silver', width=2 )
))

fig.update_layout(
    title='Alignement des cours Or et Argent ajusté avec le ratio',
    xaxis_title='Date',
    yaxis_title='Prix ($)',
    hovermode='x unified',
    template='plotly_white',
    height=600
)

fig.show()

# Calcul de l'écart entre les deux séries
df['Ecart'] = df['Or'] - df['Argent_ajuste']


In [7]:
df['Spread'] = df['Or'] - df['Argent_ajuste']
# Visualisation du spread
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Spread'], mode='lines', name='Spread'))
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(title='Spread (Or vs Argent Ajusté)', template='plotly_white')
fig.show()

In [7]:
print("\nTest de Cointégration (Augmented Dickey-Fuller):")
print("H0: Le spread n'est pas stationnaire (pas de cointégration)")
print("H1: Le spread est stationnaire (cointégration)")
result = adfuller(df['Spread'].dropna())
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print("Critical Values:")
for key, value in result[4].items():
    print(f"\t{key}: {value:.4f}")
if result[1] < 0.1:
    print("\n=> Rejet de H0 : Le spread est stationnaire. Les actifs sont cointégrés.")
    print("=> La stratégie de retour à la moyenne est valide statistiquement.")
else:
    print("\n=> Échec du rejet de H0 : Le spread n'est pas stationnaire.")


Test de Cointégration (Augmented Dickey-Fuller):
H0: Le spread n'est pas stationnaire (pas de cointégration)
H1: Le spread est stationnaire (cointégration)


KeyError: 'Spread'


la paire silver/Gold n'est donc pas cointégrée et donc stationnaire, ce qui signifie que statistiquement rien ne dis qu'un retour du spread au 0 est probable.
Il n'y a pas de fondement statistique qui puisse justifier une strategie de stat arb puisse fonctionner.

In [6]:

# Configuration
CAPITAL_INITIAL = 10000
POSITION_SIZE = 100  
FLAT_FEE = 2.0      

# Paramètres Stratégie FIXE (Hardstop)
FIXED_LEVELS = [20, 40, 60]  # Seuils d'entrée (+/-)
FIXED_STOPS = [30, 50, 70]   # Stop Loss associés (+/-)

# Paramètres Stratégie DYNAMIQUE (Z-Score)
ZSCORE_WINDOW = 50
Z_ENTRY_LEVELS = [2, 3, 4]   # Seuils d'entrée (Z-Score)
Z_STOP_LEVELS = [3, 4, 5]    # Stop Loss associés (Z-Score)

# Calcul du Z-Score
rolling_mean = df['Spread'].rolling(window=ZSCORE_WINDOW).mean()
rolling_std = df['Spread'].rolling(window=ZSCORE_WINDOW).std()
df['ZScore'] = (df['Spread'] - rolling_mean) / rolling_std

# Classe Position
class Position:
    def __init__(self, entry_price, size, side, level_idx, entry_date):
        self.entry_price = entry_price
        self.size = size
        self.side = side  # 1 for Long, -1 for Short
        self.level_idx = level_idx
        self.entry_date = entry_date

    def calculate_pnl(self, current_price):
    
    
        return (current_price - self.entry_price) * self.side * self.size

# Moteur de backtest
def run_strategy(df, strategy_type='fixed'):
    cash = CAPITAL_INITIAL
    equity = []
    positions = []
    trade_history = []
    
    for date, row in df.iterrows():
        spread_val = row['Spread']
        z_val = row['ZScore']
        
        signal_val = spread_val if strategy_type == 'fixed' else z_val
        
        # close trade tp sl
        for pos in positions[:]:
            pnl_gross = pos.calculate_pnl(spread_val)
            close_position = False
            reason = ""
            
            # Stop Loss
            if strategy_type == 'fixed':
                if pos.side == 1:  # Long
                    if spread_val <= -FIXED_STOPS[pos.level_idx]:
                        close_position = True
                        reason = "SL"
                else:  # Short
                    if spread_val >= FIXED_STOPS[pos.level_idx]:
                        close_position = True
                        reason = "SL"
            else:  # Dynamic
                if pos.side == 1:
                    if z_val <= -Z_STOP_LEVELS[pos.level_idx]:
                        close_position = True
                        reason = "SL"
                else:
                    if z_val >= Z_STOP_LEVELS[pos.level_idx]:
                        close_position = True
                        reason = "SL"
            
            # Take Profit
            if not close_position:
                if strategy_type == 'fixed':
                    if pos.side == 1 and spread_val >= 0:
                        close_position = True
                        reason = "TP"
                    elif pos.side == -1 and spread_val <= 0:
                        close_position = True
                        reason = "TP"
                else:  # Dynamic
                    if pos.side == 1 and z_val >= 0:
                        close_position = True
                        reason = "TP"
                    elif pos.side == -1 and z_val <= 0:
                        close_position = True
                        reason = "TP"
            
            if close_position:
                cash += pnl_gross - FLAT_FEE
                trade_history.append({
                    'Date': date, 'Type': 'Exit', 'Reason': reason, 
                    'PnL': pnl_gross - FLAT_FEE, 'Level': pos.level_idx, 'Side': pos.side
                })
                positions.remove(pos)
        
        # --- ENTREES ---
        levels = FIXED_LEVELS if strategy_type == 'fixed' else Z_ENTRY_LEVELS
        
        for idx, level in enumerate(levels):
            # SHORT (spread trop haut)
            if signal_val >= level:
                has_pos = any(p.level_idx == idx and p.side == -1 for p in positions)
                if not has_pos:
                    positions.append(Position(spread_val, POSITION_SIZE, -1, idx, date))
                    cash -= FLAT_FEE
                    trade_history.append({
                        'Date': date, 'Type': 'Entry', 'Side': 'Short', 
                        'Level': idx, 'Price': signal_val
                    })
            
            # LONG (spread trop bas)
            if signal_val <= -level:
                has_pos = any(p.level_idx == idx and p.side == 1 for p in positions)
                if not has_pos:
                    positions.append(Position(spread_val, POSITION_SIZE, 1, idx, date))
                    cash -= FLAT_FEE
                    trade_history.append({
                        'Date': date, 'Type': 'Entry', 'Side': 'Long', 
                        'Level': idx, 'Price': signal_val
                    })
        
        # --- VALORISATION ---
        latent_pnl = sum([p.calculate_pnl(spread_val) for p in positions])
        current_equity = cash + latent_pnl
        equity.append({'Date': date, 'Equity': current_equity})
    
    return pd.DataFrame(equity).set_index('Date'), pd.DataFrame(trade_history)

# Exécution des stratégies
print("\\n=== STRATEGIE FIXE (Hardstop) ===")
equity_fixed, trades_fixed = run_strategy(df, strategy_type='fixed')
final_fixed = equity_fixed['Equity'].iloc[-1]
pnl_fixed = final_fixed - CAPITAL_INITIAL
nb_trades_fixed = len(trades_fixed[trades_fixed['Type']=='Exit'])
print(f"Capital Final: ${final_fixed:.2f}")
print(f"PnL: ${pnl_fixed:.2f} ({pnl_fixed/CAPITAL_INITIAL*100:.2f}%)")
print(f"Nombre de trades: {nb_trades_fixed}")

print("\\n=== STRATEGIE DYNAMIQUE (Z-Score) ===")
df_dynamic = df.dropna()  # Enlever les NaN du Z-Score
equity_dynamic, trades_dynamic = run_strategy(df_dynamic, strategy_type='dynamic')
final_dynamic = equity_dynamic['Equity'].iloc[-1]
pnl_dynamic = final_dynamic - CAPITAL_INITIAL
nb_trades_dynamic = len(trades_dynamic[trades_dynamic['Type']=='Exit'])
print(f"Capital Final: ${final_dynamic:.2f}")
print(f"PnL: ${pnl_dynamic:.2f} ({pnl_dynamic/CAPITAL_INITIAL*100:.2f}%)")
print(f"Nombre de trades: {nb_trades_dynamic}")

# --- GRAPHIQUE COMPARATIF ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=equity_fixed.index, 
    y=equity_fixed['Equity'],
    mode='lines',
    name=f'Stratégie Fixe (PnL: ${pnl_fixed:.0f})',
    line=dict(color='#1f77b4', width=2.5)
))

fig.add_trace(go.Scatter(
    x=equity_dynamic.index, 
    y=equity_dynamic['Equity'],
    mode='lines',
    name=f'Stratégie Dynamique (PnL: ${pnl_dynamic:.0f})',
    line=dict(color='#ff7f0e', width=2.5)
))

fig.add_hline(y=CAPITAL_INITIAL, line_dash="dash", line_color="gray", 
              annotation_text="Capital Initial", annotation_position="right")

fig.update_layout(
    title='Comparaison PnL Cumulé: Stratégie Fixe vs Dynamique (Mean Reversion)',
    xaxis_title='Date',
    yaxis_title='Capital ($)',
    template='plotly_white',
    hovermode='x unified',
    height=600,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

KeyError: 'Spread'